<a href="https://colab.research.google.com/github/Eleonwra/elcardiocc-baseline-ner/blob/main/results/test_mbert_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Importing Dataset

In [ ]:
%pip install datasets

In [ ]:
import pickle
import datasets

In [ ]:
with open('final_test_dataset_sentences.pickle', 'rb') as data:
    final_dataset = pickle.load(data)

In [ ]:
final_dataset_sentences = {
    'test': final_dataset}

##Checking lengths

In [ ]:
from transformers import BertTokenizerFast
tokenizer = BertTokenizerFast.from_pretrained("bert-base-multilingual-cased")

In [ ]:
subset = ['test']
tokenized_data = {'test': []}
max_lengths = {name: 0 for name in subset}
sent_lengths = { 'test': []}
for section_name, section_data in final_dataset_sentences.items():
    for i in range(len(section_data)):
        texts =  ["".join(t) for t in section_data[i]["sentences_tokens"]]
        tokenized_document = tokenizer(texts)
        for j in range(len(tokenized_document['input_ids'])):
            tokenized_data[section_name].append(tokenized_document)
            sent_lengths[section_name].append(len(tokenized_document['input_ids'][j]))
            max_lengths[section_name] = max(max_lengths[section_name], len(tokenized_document['input_ids'][j]))
print("Max Length of sentences:", max_lengths)

In [ ]:
def count_elements_greater_than_threshold(values, threshold):
    return sum(1 for value in values if value > threshold)

for split in ['test']:
    threshold = 388
    count_greater_than_threshold = count_elements_greater_than_threshold(sent_lengths[split], threshold)
    print(f"Number of elements greater than {threshold} in {split}: {count_greater_than_threshold}")

##Tokenization

In [ ]:
def tokenize_and_align_labels(data_section, max_length):
    texts = ["".join(t) for t in data_section["sentences_tokens"]]
    tokenized_inputs = tokenizer(texts, padding='max_length', truncation=True, max_length=max_length)
    labels = []
    for row_idx, label_old in enumerate(data_section["sentences_ner_tags"]):
        label_new = [[] for t in tokenized_inputs.tokens(batch_index=row_idx)]
        for char_idx in range(len(data_section["sentences_tokens"][row_idx])):
            token_idx = tokenized_inputs.char_to_token(row_idx, char_idx)
            if token_idx is not None:
                label_new[token_idx].append(data_section["sentences_ner_tags"][row_idx][char_idx])
        label_new = list(map(lambda i : max(i, default=0), label_new))
        labels.append(label_new)
    tokenized_inputs["labels"] = labels + [0] * (max_length - len(labels))
    return tokenized_inputs

In [ ]:
section_name = ['test']
tokenized_data = {'test': []}
max_lengths = {
    'test': 388
}

for section_name, section_data in final_dataset_sentences.items():
    for i in range(len(section_data)):
        tokenized_document = tokenize_and_align_labels(section_data[i], max_lengths[section_name])
        tokenized_data[section_name].append(tokenized_document)

##Save

In [ ]:
with open('mBERT_tokenized_test_documents_trun.pkl', 'wb') as f:
    pickle.dump(tokenized_data, f)